In [1]:
pip install opencv-python mtcnn tqdm pandas
!pip install lz4 --extra-index-url https://python-lz4.readthedocs.io/
pip install --upgrade mtcnn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 95.3 MB/s eta 0:00:00


In [4]:
import os
import cv2
import json
import pandas as pd
from mtcnn.mtcnn import MTCNN
from tqdm.notebook import tqdm
import numpy as np

# --- 1. CONFIGURATION ---
# Base directory for the Kaggle dataset
# Adjust this path based on where you extracted your Kaggle files
KAGGLE_ROOT = '/content/drive/MyDrive/xceptionnet_data'
VIDEO_DIR = os.path.join(KAGGLE_ROOT, 'train_sample_videos') # Directory containing the video files
METADATA_PATH = os.path.join(VIDEO_DIR, 'metadata.json') # Path to the metadata file

# Output directory for the cropped faces
OUTPUT_FACES_DIR = '/content/drive/MyDrive/xceptionnet_data/cropped_faces_2'
FACE_SIZE = 299 # Target size for Xception model input

# Extraction settings
FRAMES_TO_EXTRACT = 30 # Number of frames to sample from each video
SKIP_FRAMES = 10     # Skip frames to ensure variety (e.g., sample every 10th frame)

# --- 2. SETUP AND METADATA LOADING ---

# Load metadata for labels
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

# Initialize MTCNN for face detection
detector = MTCNN()

# Create output directories
os.makedirs(os.path.join(OUTPUT_FACES_DIR, 'REAL'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_FACES_DIR, 'FAKE'), exist_ok=True)

print(f"Output directory created: {OUTPUT_FACES_DIR}")

# --- 3. FACE EXTRACTION FUNCTION ---

def extract_and_save_faces(video_path, label, video_id, max_frames=FRAMES_TO_EXTRACT, skip=SKIP_FRAMES):
    """
    Extracts faces from a video and saves them to the appropriate REAL/FAKE folder.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    frame_count = 0
    extracted_count = 0

    while cap.isOpened() and extracted_count < max_frames:
        ret, frame = cap.read()

        if not ret:
            break

        # Skip frames to sample sparsely
        if frame_count % skip == 0:
            # Convert frame to RGB for MTCNN
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Detect faces
            results = detector.detect_faces(rgb_frame)

            if results:
                # We assume the largest/most prominent face is the target
                face_data = max(results, key=lambda x: x['box'][2] * x['box'][3])
                x, y, w, h = face_data['box']

                # Add a small margin around the face to include context
                margin = int(0.3 * w)
                x1 = max(0, x - margin)
                y1 = max(0, y - margin)
                x2 = min(frame.shape[1], x + w + margin)
                y2 = min(frame.shape[0], y + h + margin)

                # Crop the face
                cropped_face = frame[y1:y2, x1:x2]

                # Resize the face to the target input size (Xception default is 299)
                resized_face = cv2.resize(cropped_face, (FACE_SIZE, FACE_SIZE))

                # Determine output folder
                output_folder = os.path.join(OUTPUT_FACES_DIR, label)

                # Save the image
                filename = f"{video_id}_{extracted_count:03d}.jpg"
                cv2.imwrite(os.path.join(output_folder, filename), resized_face)

                extracted_count += 1

        frame_count += 1

    cap.release()
    print(f"Extracted {extracted_count} faces from {video_id}")


# --- 4. MAIN PROCESSING LOOP ---

video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')]

for video_file in tqdm(video_files, desc="Processing Videos"):
    video_id = video_file
    video_path = os.path.join(VIDEO_DIR, video_file)

    # Get label from metadata
    if video_id in metadata:
        label = metadata[video_id]['label']
    else:
        # Skip video if no label is found (shouldn't happen with the sample data)
        print(f"Skipping {video_id}: Label not found.")
        continue

    extract_and_save_faces(video_path, label, video_id)

print("\nFace extraction complete!")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Output directory created: /content/drive/MyDrive/xceptionnet_data/cropped_faces_2


Processing Videos:   0%|          | 0/400 [00:00<?, ?it/s]

Extracted 30 faces from aapnvogymq.mp4
Extracted 30 faces from aagfhgtpmv.mp4
Extracted 30 faces from abqwwspghj.mp4
Extracted 30 faces from acqfdwsrhi.mp4
Extracted 30 faces from acxnxvbsxk.mp4
Extracted 30 faces from abarnvbtwb.mp4
Extracted 26 faces from abofeumbvv.mp4
Extracted 30 faces from acifjvzvpm.mp4
Extracted 23 faces from adhsbajydo.mp4
Extracted 30 faces from aczrgyricp.mp4
Extracted 30 faces from acxwigylke.mp4
Extracted 30 faces from adohikbdaz.mp4
Extracted 30 faces from adylbeequz.mp4
Extracted 30 faces from aettqgevhz.mp4
Extracted 30 faces from afoovlsmtx.mp4
Extracted 30 faces from aelzhcnwgf.mp4
Extracted 30 faces from aevrfsexku.mp4
Extracted 30 faces from aelfnikyqj.mp4
Extracted 30 faces from ahbweevwpv.mp4
Extracted 29 faces from agrmhtjdlk.mp4
Extracted 30 faces from agqphdxmwt.mp4
Extracted 30 faces from agdkmztvby.mp4
Extracted 30 faces from ahqqqilsxt.mp4
Extracted 30 faces from aipfdnwpoo.mp4
Extracted 30 faces from ahfazfbntc.mp4
Extracted 30 faces from a

In [3]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 143.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 188.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00


In [2]:
# Install lz4 explicitly before importing MTCNN
!pip install lz4 --extra-index-url https://python-lz4.readthedocs.io/

Looking in indexes: https://pypi.org/simple, https://python-lz4.readthedocs.io/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.4 MB/s eta 0:00:00


In [3]:
pip install --upgrade mtcnn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.0 MB/s eta 0:00:00


In [6]:
pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2
